# Fashion-MNIST Image Classification with CNN
### Convolutional Neural Network with iterative improvement based on confusion matrix diagnosis

**Dataset:** [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) — 70,000 grayscale 28x28 images across 10 clothing categories (60,000 train / 10,000 test), built into `tf.keras.datasets`.

**Approach:** build a baseline CNN, diagnose its errors using a confusion matrix (not just accuracy), then make targeted architecture changes based on exactly where the model struggles — rather than blindly tuning hyperparameters.

**Structure:**
1. EDA
2. Preprocessing + data augmentation
3. Baseline CNN — architecture, training, evaluation
4. Diagnosis via confusion matrix
5. Improved CNN — targeted changes based on the diagnosis
6. Before/after comparison
7. Key findings

**Requirements:** `tensorflow`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`


## 1. EDA

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Dense, Conv2D, MaxPool2D, Flatten, Dropout,
                                       BatchNormalization, RandomFlip, RandomRotation, RandomZoom)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Pixel value range:", X_train.min(), "-", X_train.max())


In [ ]:
# Random sample of the actual dataset
plt.figure(figsize=(10, 10))
np.random.seed(42)
random_idx = np.random.choice(len(X_train), 25, replace=False)
for i, idx in enumerate(random_idx):
    plt.subplot(5, 5, i+1)
    plt.imshow(X_train[idx], cmap='gray')
    plt.title(class_names[y_train[idx]], fontsize=9)
    plt.axis('off')
plt.suptitle('Random Sample of Fashion-MNIST', fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()


In [ ]:
# One clear example per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(class_names[i], fontweight='bold')
    ax.axis('off')
plt.suptitle('One Example per Class', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
unique, counts = np.unique(y_train, return_counts=True)
colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar([class_names[i] for i in unique], counts, color=colors, edgecolor='white')
ax.set_title('Class Distribution — Fashion-MNIST', fontweight='bold')
ax.set_ylabel('Number of Images')
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_train.flatten(), bins=50, color='#4A90D9', edgecolor='white')
axes[0].set_title('Overall Pixel Intensity Distribution', fontweight='bold')
axes[0].set_xlabel('Pixel value (0-255)')
axes[0].set_ylabel('Frequency')

mean_intensity_per_class = [X_train[y_train == i].mean() for i in range(10)]
axes[1].bar(class_names, mean_intensity_per_class, color=colors, edgecolor='white')
axes[1].set_title('Mean Pixel Intensity by Class', fontweight='bold')
axes[1].set_xticklabels(class_names, rotation=30, ha='right')

plt.tight_layout()
plt.show()


## 2. Preprocessing + data augmentation

Pixel values are normalized to 0-1 and reshaped to add the channel dimension (grayscale = 1 channel). Augmentation layers run only during training (`.fit()`) and are automatically skipped during evaluation/prediction — no separate train/test generator objects needed, unlike `ImageDataGenerator`.


In [ ]:
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

X_train_norm = X_train_norm.reshape(-1, 28, 28, 1)
X_test_norm = X_test_norm.reshape(-1, 28, 28, 1)

print("Train shape:", X_train_norm.shape, "Test shape:", X_test_norm.shape)


In [ ]:
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),   # mirrors left-right — fine for clothing (unlike digits/text)
    RandomRotation(0.08),       # closest layer-based equivalent to shear
    RandomZoom(0.2),
])


## 3. Baseline CNN — architecture, training, evaluation

In [ ]:
cnn = Sequential([
    Input(shape=(28, 28, 1)),
    data_augmentation,
    Conv2D(filters=32, kernel_size=3, activation='relu'),
    MaxPool2D(pool_size=2, strides=2),
    Conv2D(filters=64, kernel_size=3, activation='relu'),
    MaxPool2D(pool_size=2, strides=2),
    Flatten(),
    Dense(units=128, activation='relu'),
    Dropout(0.3),
    Dense(units=10, activation='softmax')
])

cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

history = cnn.fit(
    X_train_norm, y_train,
    validation_data=(X_test_norm, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train', color='#4A90D9', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', color='#E76F6F', linewidth=2, linestyle='--')
axes[0].set_title('Loss (Baseline)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train', color='#4A90D9', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation', color='#E76F6F', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy (Baseline)', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
test_loss, test_acc = cnn.evaluate(X_test_norm, y_test, verbose=0)
print(f"Baseline test accuracy: {test_acc:.4f}")
baseline_acc = test_acc

y_pred_probs = cnn.predict(X_test_norm)
y_pred_baseline = np.argmax(y_pred_probs, axis=1)

baseline_report = classification_report(y_test, y_pred_baseline, target_names=class_names, output_dict=True)
print(classification_report(y_test, y_pred_baseline, target_names=class_names))


## 4. Diagnosis via confusion matrix

Accuracy alone doesn't say *where* the model is wrong. The confusion matrix does.


In [ ]:
cm_baseline = confusion_matrix(y_test, y_pred_baseline)

plt.figure(figsize=(9, 8))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Baseline Model', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**Diagnosis:** errors are not randomly distributed. They concentrate in a 4-way cluster of visually similar upper-body garments: **Shirt, T-shirt/top, Pullover, and Coat** are repeatedly confused with each other, while footwear (Sneaker/Sandal/Ankle boot), Trouser, and Bag are classified with 93-98% F1 — these have distinctive silhouettes the model separates cleanly. `Shirt` is the worst class by a clear margin.

This — not the overall accuracy number — is what motivates the architecture changes below: more capacity specifically aimed at distinguishing subtle shape differences (collar, sleeve, fit) rather than a blind hyperparameter search.


## 5. Improved CNN — targeted changes based on the diagnosis

Changes made, each targeting the diagnosed weakness:
- **A third Conv2D block (128 filters) + `padding='same'`** — more capacity to learn finer shape distinctions, without shrinking spatial dimensions too fast
- **BatchNormalization** after every conv layer — stabilizes and speeds up training
- **Lower learning rate (0.0005) + `ReduceLROnPlateau`** — finer convergence instead of overshooting
- **More epochs (30) with higher patience** — augmentation makes each epoch harder, so the model needs more room to converge


In [ ]:
cnn = Sequential([
    Input(shape=(28, 28, 1)),
    data_augmentation,

    Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=2, strides=2),

    Conv2D(filters=64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=2, strides=2),

    Conv2D(filters=128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=2, strides=2),

    Flatten(),
    Dense(units=256, activation='relu'),
    Dropout(0.4),
    Dense(units=10, activation='softmax')
])

cnn.compile(optimizer=Adam(learning_rate=0.0005),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])
cnn.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

history = cnn.fit(
    X_train_norm, y_train,
    validation_data=(X_test_norm, y_test),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)


In [ ]:
test_loss, test_acc = cnn.evaluate(X_test_norm, y_test, verbose=0)
print(f"Improved test accuracy: {test_acc:.4f}")
improved_acc = test_acc

y_pred_probs = cnn.predict(X_test_norm)
y_pred_improved = np.argmax(y_pred_probs, axis=1)

improved_report = classification_report(y_test, y_pred_improved, target_names=class_names, output_dict=True)
print(classification_report(y_test, y_pred_improved, target_names=class_names))


In [ ]:
cm_improved = confusion_matrix(y_test, y_pred_improved)

plt.figure(figsize=(9, 8))
sns.heatmap(cm_improved, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Improved Model', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6. Before / after comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Class': class_names,
    'Baseline F1': [baseline_report[c]['f1-score'] for c in class_names],
    'Improved F1': [improved_report[c]['f1-score'] for c in class_names],
})
comparison['Change'] = (comparison['Improved F1'] - comparison['Baseline F1']).round(3)
comparison = comparison.round(3).sort_values('Change', ascending=False)

print(f"Baseline accuracy: {baseline_acc:.4f}  |  Improved accuracy: {improved_acc:.4f}")
comparison


## 7. Key findings

- Test accuracy improved from **~87.8% (baseline) to ~91.4% (improved)** after targeted architecture changes
- The largest F1 gains landed almost entirely on the classes diagnosed as weakest — **Shirt, Coat, Pullover** — confirming the changes addressed the actual problem rather than improving everything uniformly
- Classes that were already easy (Trouser, Bag, footwear) saw little to no change — they were already near their ceiling
- **Shirt remains the hardest class** even after improvement — squeezing further would likely require transfer learning (a pretrained model fine-tuned on this data) rather than more tweaking of this from-scratch architecture

## Limitations
- This is a from-scratch CNN on low-resolution (28x28) grayscale images — a deeper architecture or transfer learning would likely push accuracy further, particularly on the Shirt/Pullover/Coat boundary
- No test-time augmentation or ensembling was used, both common ways to squeeze out additional accuracy at the cost of inference time
